In [0]:
%run "/Workspace/Users/rahulpatel@cyntexa.com/DataEngineering-Project/de_project/src/includes"

In [0]:
# dbutils.widgets.text("catalog","de_dev")

In [0]:
%sql
CREATE OR REPLACE TABLE ${catalog}.gold.customer_kpi AS

WITH customer_data AS (

    SELECT
        c.customer_id,
        c.name,
        s.sale_id,
        s.sale_date,
        s.sale_amount,

        -- New vs Returning Customer
        CASE
            WHEN s.sale_date = MIN(s.sale_date)
                 OVER(PARTITION BY c.customer_id)
            THEN 'New Customer'
            ELSE 'Returning Customer'
        END AS customer_type

    FROM ${catalog}.silver.customers_scd_1 c

    JOIN ${catalog}.silver.sales_scd_1 s
        ON c.customer_id = s.customer_id

)

SELECT

    customer_id,

    name,

    customer_type,

    sale_date,

    sale_amount,

    -- Customer Lifetime Value (CLV)
    SUM(sale_amount)
        OVER(PARTITION BY customer_id)
        AS customer_lifetime_value,

    -- Average Order Value by Customer Type
    AVG(sale_amount)
        OVER(PARTITION BY customer_type)
        AS avg_order_value,

    -- Previous Order Amount
    LAG(sale_amount)
        OVER(
            PARTITION BY customer_id
            ORDER BY sale_date
        ) AS previous_order,

    -- Next Order Amount
    LEAD(sale_amount)
        OVER(
            PARTITION BY customer_id
            ORDER BY sale_date
        ) AS next_order,

    -- First Purchase Amount
    FIRST_VALUE(sale_amount)
        OVER(
            PARTITION BY customer_id
            ORDER BY sale_date
        ) AS first_purchase_amount

FROM customer_data;